In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from xgboost import XGBClassifier


# ============================================================
# 0. Path configuration
# ============================================================

def find_project_root():
    if "__file__" in globals():
        start_dir = Path(__file__).resolve().parent
    else:
        start_dir = Path.cwd().resolve()

    for path in [start_dir] + list(start_dir.parents):
        if (path / "Code").exists() and (path / "Data").exists() and (path / "Result").exists():
            return path

    raise FileNotFoundError(
        "Project root was not found. Please make sure the project contains Code, Data, and Result folders."
    )

PROJECT_ROOT = find_project_root()

# Base directory containing the numbered result folders.
RESULT_BASE_DIR = PROJECT_ROOT / "Result"

# Fixed five-fold train/test datasets generated by Script 01.
DATA_ROOT = RESULT_BASE_DIR / "01_Foldwise_Preprocessing_and_Ratio_Features"

# Automatically locate the Script 04 and Script 05 result folders.
CHAMPION_DIR_PREFIX = "04_Foldwise_Correlation_Clustering_Champion_Selection_Rho_Sensitivity"
STABILITY_DIR_PREFIX = "05_Stable_Cluster_Champions_and_Novel_Ratio_Candidates"

# Output directory for this script.
OUT_DIR = RESULT_BASE_DIR / "06_Stable_Novel_Ratio_Group_Contribution_Experiment"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TYPE_COL = "Type"

NON_NUM_COLS = {
    "No.",
    "Samp1e",
    "Type",
    "Reference"
}

IMPUTATION_METHODS = ["global_mean", "knn"]
N_OUTER_FOLDS = 5
RHO_TAG = "0.90"

SEED = 42

# Whether to additionally output exploratory results based on the global
# stable novel-ratio list.
#
# Note:
# The primary leakage-controlled performance conclusions should be based on
# fold-wise novel champions. The global stable-ratio list is exploratory.
INCLUDE_GLOBAL_STABLE_RATIO_EXPLORATORY = True

# Minimum appearance count for selecting stable novel ratios from the
# Script 05 summary table.
GLOBAL_STABLE_MIN_APPEARANCE = 6

# Whether to use GPU for XGBoost.
# For most local Windows environments, keep this as False.
# For CUDA-enabled servers, this can be set to True.
USE_GPU = False

XGB_PARAMS = dict(
    n_estimators=900,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.75,
    gamma=0.0,
    reg_lambda=6.0,
    reg_alpha=0.0,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=SEED,
    n_jobs=-1,
    tree_method="hist",
)

if USE_GPU:
    XGB_PARAMS["device"] = "cuda"


# ============================================================
# 1. Utility functions
# ============================================================

def find_dir_by_prefix(base_dir, prefix):
    """
    Automatically find a folder under base_dir whose name starts with prefix.
    """
    if not base_dir.exists():
        raise FileNotFoundError(f"Directory does not exist: {base_dir}")

    candidates = []
    for path in base_dir.iterdir():
        if path.is_dir() and path.name.startswith(prefix):
            candidates.append(path)

    if len(candidates) == 0:
        raise FileNotFoundError(
            f"No folder starting with '{prefix}' was found under: {base_dir}"
        )

    if len(candidates) > 1:
        print(f"Warning: multiple folders starting with '{prefix}' were found. The first one will be used:")
        for candidate in candidates:
            print("  ", candidate)

    return candidates[0]


def build_model():
    return XGBClassifier(**XGB_PARAMS)


def clean_display_name(name):
    """
    Clean feature names for figures and output display only.

    Original column names are not modified.
    """
    s = str(name)
    s = s.replace("10000*Ga/A1", "10000xGa/Al")
    s = s.replace("10000*Ga/Al", "10000xGa/Al")
    s = s.replace("10000xGa/Al", "10000xGa/Al")
    s = s.replace("A12O3", "Al2O3")
    s = s.replace("R_Major_", "")
    s = s.replace("R_Trace_", "")
    return s


def is_constructed_ratio(feature):
    s = str(feature)
    return s.startswith("R_Major_") or s.startswith("R_Trace_")


def is_classical_feature(feature):
    """
    Mark traditional or classical indicators.

    This flag is used only for feature grouping. It does not imply that
    all other features are necessarily new geological indicators.
    """
    s = str(feature)

    classical_exact = {
        "A/CNK",
        "A/NK",
        "10000*Ga/Al",
        "10000*Ga/A1",
        "10000xGa/Al",
        "Zr+Nb+Ce+Y",
        "R_Trace_Nb/Ta",
        "R_Trace_Zr/Hf",
        "R_Trace_Sr/Y",
        "R_Major_Fe2O3t/MgO",
        "R_Major_Na2O/K2O",
        "R_Major_K2O/Na2O",
    }

    if s in classical_exact:
        return True

    patterns = [
        "A/CNK",
        "A/NK",
        "10000*Ga/Al",
        "10000*Ga/A1",
        "10000xGa/Al",
        "Zr+Nb+Ce+Y",
        "Nb/Ta",
        "Zr/Hf",
        "Sr/Y",
        "Fe2O3t/MgO",
        "K2O/Na2O",
        "Na2O/K2O",
    ]

    return any(p in s for p in patterns)


def is_candidate_novel_ratio(feature):
    """
    Define candidate novel ratios as systematically constructed ratios
    that are not common classical indicators.
    """
    return is_constructed_ratio(feature) and (not is_classical_feature(feature))


def ratio_group(feature):
    """
    Assign a simple geochemical group to a ratio feature.
    """
    s = str(feature)

    if not is_constructed_ratio(s):
        return "Original/classical feature"

    if s.startswith("R_Major_"):
        return "Major-element ratio"

    body = s.replace("R_Trace_", "")

    hfse = {"Nb", "Ta", "Zr", "Hf", "Ti", "Y"}
    ree = {
        "La", "Ce", "Pr", "Nd", "Sm", "Eu", "Gd",
        "Tb", "Dy", "Ho", "Er", "Tm", "Yb", "Lu"
    }
    lile = {"Rb", "Sr", "Ba", "Cs", "Pb"}
    th_u = {"Th", "U"}

    parts = re.split(r"[/_+\-*()]+", body)
    parts = [p for p in parts if p]

    has_hfse = any(p in hfse for p in parts)
    has_ree = any(p in ree for p in parts)
    has_lile = any(p in lile for p in parts)
    has_thu = any(p in th_u for p in parts)

    if has_hfse and has_ree:
        return "HFSE-REE ratio"
    if has_hfse and has_thu:
        return "HFSE-Th-U ratio"
    if has_hfse:
        return "HFSE ratio"
    if has_ree:
        return "REE ratio"
    if has_lile:
        return "LILE-related ratio"

    return "Trace-element ratio"


def get_feature_cols_from_train(train_df):
    """
    Determine feature columns only from the training fold.
    """
    candidate_cols = [
        c for c in train_df.columns
        if c not in NON_NUM_COLS
    ]

    X_train = train_df[candidate_cols].copy()

    for c in X_train.columns:
        X_train[c] = pd.to_numeric(X_train[c], errors="coerce")

    X_train = X_train.replace([np.inf, -np.inf], np.nan)

    X_train = X_train.loc[:, X_train.notna().any(axis=0)]
    X_train = X_train.loc[:, X_train.nunique(dropna=True) > 1]

    return X_train.columns.tolist()


def prepare_xy(train_df, test_df):
    """
    Construct X and y for the training and test folds.
    """
    if TYPE_COL not in train_df.columns or TYPE_COL not in test_df.columns:
        raise ValueError(f"Label column was not found: {TYPE_COL}")

    feature_cols = get_feature_cols_from_train(train_df)

    X_train = train_df[feature_cols].copy()
    X_test = test_df[feature_cols].copy()

    for c in feature_cols:
        X_train[c] = pd.to_numeric(X_train[c], errors="coerce")
        X_test[c] = pd.to_numeric(X_test[c], errors="coerce")

    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    X_test = X_test.replace([np.inf, -np.inf], np.nan)

    if X_train.isna().sum().sum() > 0:
        raise ValueError(
            "NaN values remain in X_train. Please check the previous "
            "fold-wise preprocessing step."
        )

    if X_test.isna().sum().sum() > 0:
        raise ValueError(
            "NaN values remain in X_test. Please check the previous "
            "fold-wise preprocessing step."
        )

    y_train_raw = train_df[TYPE_COL].astype(str).values
    y_test_raw = test_df[TYPE_COL].astype(str).values

    le = LabelEncoder()
    y_train = le.fit_transform(y_train_raw)
    y_test = le.transform(y_test_raw)

    return (
        X_train.reset_index(drop=True),
        X_test.reset_index(drop=True),
        y_train,
        y_test,
        le,
        feature_cols
    )


def evaluate_feature_set(
    X_train,
    y_train,
    X_test,
    y_test,
    label_encoder,
    features,
    feature_set_name
):
    """
    Train and evaluate one feature set.
    """
    features = [f for f in features if f in X_train.columns]

    if len(features) == 0:
        return None, None, None

    model = build_model()
    model.fit(X_train[features], y_train)

    pred = model.predict(X_test[features])

    metrics = {
        "Feature_set": feature_set_name,
        "N_features": len(features),
        "Accuracy": float(accuracy_score(y_test, pred)),
        "Balanced_accuracy": float(balanced_accuracy_score(y_test, pred)),
        "Macro_precision": float(
            precision_score(y_test, pred, average="macro", zero_division=0)
        ),
        "Macro_recall": float(
            recall_score(y_test, pred, average="macro", zero_division=0)
        ),
        "Macro_F1": float(
            f1_score(y_test, pred, average="macro", zero_division=0)
        ),
    }

    class_rows = []

    labels = list(range(len(label_encoder.classes_)))
    p_cls = precision_score(
        y_test,
        pred,
        average=None,
        labels=labels,
        zero_division=0
    )
    r_cls = recall_score(
        y_test,
        pred,
        average=None,
        labels=labels,
        zero_division=0
    )
    f_cls = f1_score(
        y_test,
        pred,
        average=None,
        labels=labels,
        zero_division=0
    )

    for i, cls_name in enumerate(label_encoder.classes_):
        class_rows.append({
            "Feature_set": feature_set_name,
            "Class": cls_name,
            "Precision": float(p_cls[i]),
            "Recall": float(r_cls[i]),
            "F1": float(f_cls[i])
        })

    cm = confusion_matrix(y_test, pred, labels=labels)
    cm_df = pd.DataFrame(
        cm,
        index=[f"True_{c}" for c in label_encoder.classes_],
        columns=[f"Pred_{c}" for c in label_encoder.classes_]
    )
    cm_df.insert(0, "Feature_set", feature_set_name)

    return metrics, pd.DataFrame(class_rows), cm_df


# ============================================================
# 2. Read Script 04 and Script 05 results
# ============================================================

def find_champion_xlsx(champion_root, method, fold, rho_tag="0.90"):
    """
    Find the champion result file for a given method, outer fold,
    and rho threshold.
    """
    rho_dir = (
        champion_root
        / method
        / f"outer_fold_{fold:02d}"
        / f"rho_{rho_tag}"
    )

    if not rho_dir.exists():
        raise FileNotFoundError(f"Rho directory was not found: {rho_dir}")

    expected = (
        rho_dir
        / f"outer_fold_{fold:02d}_cluster_champions_SHAP_innerCV_rho{rho_tag}.xlsx"
    )

    if expected.exists():
        return expected

    candidates = [
        p for p in rho_dir.iterdir()
        if p.is_file() and p.suffix.lower() == ".xlsx" and "cluster_champions" in p.name
    ]

    if len(candidates) == 0:
        raise FileNotFoundError(f"No champion xlsx file was found in: {rho_dir}")

    return candidates[0]


def read_champions(champion_xlsx):
    """
    Read the ClusterChampions sheet.
    """
    champ_df = pd.read_excel(champion_xlsx, sheet_name="ClusterChampions")

    if "champion" not in champ_df.columns:
        raise ValueError(
            f"The ClusterChampions sheet has no 'champion' column: {champion_xlsx}"
        )

    champions = champ_df["champion"].astype(str).tolist()

    return champions, champ_df


def find_stability_summary_file(stability_root):
    """
    Find the Script 05 summary workbook.
    """
    if not stability_root.exists():
        print(f"Warning: Script 05 summary directory was not found: {stability_root}")
        return None

    candidates = [
        p for p in stability_root.iterdir()
        if p.is_file() and p.suffix.lower() == ".xlsx" and "stable_champions" in p.name
    ]

    if len(candidates) == 0:
        candidates = [
            p for p in stability_root.iterdir()
            if p.is_file() and p.suffix.lower() == ".xlsx"
        ]

    if len(candidates) == 0:
        print(f"Warning: no xlsx file was found in Script 05 summary directory: {stability_root}")
        return None

    return candidates[0]


def read_global_stable_ratio_candidates(stability_root):
    """
    Read stable_ratio_candidates from the Script 05 summary.

    This is used only for exploratory supplementary analysis, not for
    the primary leakage-controlled performance conclusion.
    """
    summary_file = find_stability_summary_file(stability_root)

    if summary_file is None:
        return []

    print(f"Reading stable candidate-ratio summary from Script 05: {summary_file}")

    xls = pd.ExcelFile(summary_file)

    if "stable_ratio_candidates" not in xls.sheet_names:
        print("Warning: the stable_ratio_candidates sheet was not found.")
        return []

    df = pd.read_excel(summary_file, sheet_name="stable_ratio_candidates")

    if "Feature" not in df.columns:
        print("Warning: the stable_ratio_candidates sheet has no Feature column.")
        return []

    if "Appearance_count" in df.columns:
        df = df[df["Appearance_count"] >= GLOBAL_STABLE_MIN_APPEARANCE].copy()

    features = df["Feature"].astype(str).tolist()

    return features


# ============================================================
# 3. Build different feature sets
# ============================================================

def build_classical_features(all_feature_cols):
    """
    Extract traditional or classical indicators from the current fold.
    """
    return [f for f in all_feature_cols if is_classical_feature(f)]


def build_feature_sets(all_feature_cols, champions, global_stable_ratios=None):
    """
    Build feature sets compared in the current fold.
    """
    all_feature_cols = list(all_feature_cols)
    champions = [f for f in champions if f in all_feature_cols]

    classical = build_classical_features(all_feature_cols)

    # Fold-wise novel champion ratios are used for the primary
    # leakage-controlled analysis.
    fold_novel_champion_ratios = [
        f for f in champions
        if is_candidate_novel_ratio(f)
    ]

    champion_without_fold_novel = [
        f for f in champions
        if f not in set(fold_novel_champion_ratios)
    ]

    classical_plus_fold_novel = sorted(
        set(classical).union(fold_novel_champion_ratios)
    )

    feature_sets = {
        "Full_features": all_feature_cols,
        "Champions_rho0.90": champions,
        "Classical_features_only": classical,
        "Fold_novel_champion_ratios_only": fold_novel_champion_ratios,
        "Classical_plus_fold_novel_champions": classical_plus_fold_novel,
        "Champions_without_fold_novel_ratios": champion_without_fold_novel,
    }

    # Exploratory feature sets based on the global stable novel-ratio list.
    if INCLUDE_GLOBAL_STABLE_RATIO_EXPLORATORY and global_stable_ratios:
        global_stable_ratios_present = [
            f for f in global_stable_ratios
            if f in all_feature_cols
        ]

        classical_plus_global_stable = sorted(
            set(classical).union(global_stable_ratios_present)
        )

        feature_sets["Global_stable_ratio_candidates_only_exploratory"] = global_stable_ratios_present
        feature_sets["Classical_plus_global_stable_ratios_exploratory"] = classical_plus_global_stable

    return feature_sets


def feature_set_detail_rows(method, fold, feature_sets):
    rows = []

    for fs_name, feats in feature_sets.items():
        for f in feats:
            rows.append({
                "Method": method,
                "Outer_fold": fold,
                "Feature_set": fs_name,
                "Feature": f,
                "Display_feature": clean_display_name(f),
                "Is_constructed_ratio": is_constructed_ratio(f),
                "Is_classical_feature": is_classical_feature(f),
                "Is_candidate_novel_ratio": is_candidate_novel_ratio(f),
                "Ratio_group": ratio_group(f)
            })

    return rows


# ============================================================
# 4. Summary and statistical tests
# ============================================================

def mean_std_text(x):
    x = pd.to_numeric(x, errors="coerce")
    if x.notna().sum() == 0:
        return ""
    return f"{x.mean():.4f} +/- {x.std(ddof=1):.4f}"


def summarize_metrics(metrics_df):
    """
    Summarize outer-fold performance by Method and Feature_set.
    """
    metric_cols = [
        "N_features",
        "Accuracy",
        "Balanced_accuracy",
        "Macro_precision",
        "Macro_recall",
        "Macro_F1"
    ]

    agg_dict = {}

    for c in metric_cols:
        agg_dict[f"{c}_mean"] = (c, "mean")
        agg_dict[f"{c}_std"] = (c, "std")
        agg_dict[f"{c}_mean_plus_minus_std"] = (c, mean_std_text)

    summary = (
        metrics_df
        .groupby(["Method", "Feature_set"], as_index=False)
        .agg(**agg_dict)
    )

    return summary


def summarize_classwise(classwise_df):
    """
    Summarize class-wise metrics by Method, Feature_set, and Class.
    """
    agg_dict = {
        "Precision_mean": ("Precision", "mean"),
        "Precision_std": ("Precision", "std"),
        "Precision_mean_plus_minus_std": ("Precision", mean_std_text),
        "Recall_mean": ("Recall", "mean"),
        "Recall_std": ("Recall", "std"),
        "Recall_mean_plus_minus_std": ("Recall", mean_std_text),
        "F1_mean": ("F1", "mean"),
        "F1_std": ("F1", "std"),
        "F1_mean_plus_minus_std": ("F1", mean_std_text),
    }

    summary = (
        classwise_df
        .groupby(["Method", "Feature_set", "Class"], as_index=False)
        .agg(**agg_dict)
    )

    return summary


def paired_delta_tests(metrics_df):
    """
    Perform paired tests for key feature-set differences.
    """
    comparisons = [
        (
            "Classical_plus_fold_novel_champions",
            "Classical_features_only",
            "Classical_plus_fold_novel_minus_Classical"
        ),
        (
            "Champions_rho0.90",
            "Champions_without_fold_novel_ratios",
            "Champions_minus_Champions_without_novel"
        ),
        (
            "Fold_novel_champion_ratios_only",
            "Classical_features_only",
            "Novel_only_minus_Classical"
        ),
    ]

    if INCLUDE_GLOBAL_STABLE_RATIO_EXPLORATORY:
        comparisons.extend([
            (
                "Classical_plus_global_stable_ratios_exploratory",
                "Classical_features_only",
                "Classical_plus_global_stable_minus_Classical_exploratory"
            ),
            (
                "Global_stable_ratio_candidates_only_exploratory",
                "Classical_features_only",
                "Global_stable_only_minus_Classical_exploratory"
            )
        ])

    rows = []

    metric_list = ["Accuracy", "Balanced_accuracy", "Macro_F1"]

    for method in metrics_df["Method"].unique():
        sub = metrics_df[metrics_df["Method"] == method].copy()

        for fs_a, fs_b, comp_name in comparisons:
            for metric in metric_list:
                a = (
                    sub[sub["Feature_set"] == fs_a][["Outer_fold", metric]]
                    .rename(columns={metric: "A"})
                )
                b = (
                    sub[sub["Feature_set"] == fs_b][["Outer_fold", metric]]
                    .rename(columns={metric: "B"})
                )

                merged = a.merge(b, on="Outer_fold", how="inner")

                if len(merged) == 0:
                    continue

                delta = merged["A"] - merged["B"]

                mean_delta = float(delta.mean())
                sd_delta = float(delta.std(ddof=1)) if len(delta) > 1 else np.nan

                if len(delta) >= 2:
                    try:
                        t_stat, t_p = stats.ttest_1samp(delta.values, popmean=0.0)
                    except Exception:
                        t_stat, t_p = np.nan, np.nan

                    try:
                        if np.allclose(delta.values, 0):
                            w_stat, w_p = 0.0, 1.0
                        else:
                            w_stat, w_p = stats.wilcoxon(delta.values)
                    except Exception:
                        w_stat, w_p = np.nan, np.nan
                else:
                    t_stat, t_p, w_stat, w_p = np.nan, np.nan, np.nan, np.nan

                rows.append({
                    "Method": method,
                    "Comparison": comp_name,
                    "Metric": metric,
                    "N_pairs": len(delta),
                    "Mean_delta": mean_delta,
                    "SD_delta": sd_delta,
                    "Delta_values": ",".join([f"{v:.6f}" for v in delta.values]),
                    "Paired_t_stat": float(t_stat) if pd.notna(t_stat) else np.nan,
                    "Paired_t_p": float(t_p) if pd.notna(t_p) else np.nan,
                    "Wilcoxon_stat": float(w_stat) if pd.notna(w_stat) else np.nan,
                    "Wilcoxon_p": float(w_p) if pd.notna(w_p) else np.nan,
                })

    return pd.DataFrame(rows)


# ============================================================
# 5. Plotting functions
# ============================================================

def plot_macro_f1_summary(summary_df, out_path):
    """
    Plot mean +/- standard deviation of Macro-F1 values.
    """
    if summary_df.empty:
        return

    order = [
        "Full_features",
        "Champions_rho0.90",
        "Classical_features_only",
        "Fold_novel_champion_ratios_only",
        "Classical_plus_fold_novel_champions",
        "Champions_without_fold_novel_ratios",
        "Global_stable_ratio_candidates_only_exploratory",
        "Classical_plus_global_stable_ratios_exploratory",
    ]

    plot_df = summary_df.copy()
    plot_df["Feature_set"] = pd.Categorical(
        plot_df["Feature_set"],
        categories=order,
        ordered=True
    )
    plot_df = plot_df.sort_values(["Method", "Feature_set"])

    methods = plot_df["Method"].unique().tolist()

    for method in methods:
        sub = plot_df[plot_df["Method"] == method].dropna(subset=["Feature_set"]).copy()

        plt.figure(figsize=(12, 6), dpi=300)

        x = np.arange(len(sub))
        y = sub["Macro_F1_mean"].values
        yerr = sub["Macro_F1_std"].values

        plt.bar(x, y, yerr=yerr, capsize=4)

        plt.xticks(
            x,
            sub["Feature_set"].astype(str),
            rotation=35,
            ha="right",
            fontsize=9,
            fontweight="bold"
        )

        plt.ylabel("Outer-test Macro-F1", fontsize=12, fontweight="bold")
        plt.title(
            f"Feature-Set Contribution Analysis ({method})",
            fontsize=14,
            fontweight="bold"
        )
        plt.ylim(0, max(1.0, np.nanmax(y + yerr) + 0.05))
        plt.tight_layout()

        method_out = out_path.with_name(f"{out_path.stem}_{method}{out_path.suffix}")
        plt.savefig(method_out, bbox_inches="tight")
        plt.close()


def plot_delta_summary(delta_df, out_path):
    """
    Plot key Macro-F1 differences.
    """
    if delta_df.empty:
        return

    plot_df = delta_df[delta_df["Metric"] == "Macro_F1"].copy()

    if plot_df.empty:
        return

    plot_df = plot_df.sort_values(["Method", "Comparison"])

    plt.figure(figsize=(12, 6), dpi=300)

    labels = [
        f"{r.Method}\n{r.Comparison}"
        for r in plot_df.itertuples()
    ]

    x = np.arange(len(plot_df))

    plt.bar(
        x,
        plot_df["Mean_delta"].values,
        yerr=plot_df["SD_delta"].values,
        capsize=4
    )

    plt.axhline(0, linewidth=1)

    plt.xticks(
        x,
        labels,
        rotation=35,
        ha="right",
        fontsize=8,
        fontweight="bold"
    )

    plt.ylabel("Mean Delta Macro-F1", fontsize=12, fontweight="bold")
    plt.title(
        "Paired Feature-Set Contribution Differences",
        fontsize=14,
        fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()


# ============================================================
# 6. Main workflow
# ============================================================

def main():
    warnings.filterwarnings("ignore")

    champion_root = find_dir_by_prefix(RESULT_BASE_DIR, CHAMPION_DIR_PREFIX)
    stability_root = find_dir_by_prefix(RESULT_BASE_DIR, STABILITY_DIR_PREFIX)

    print("Script 04 result directory:", champion_root)
    print("Script 05 summary directory:", stability_root)
    print("Fixed five-fold data directory:", DATA_ROOT)
    print("Output directory:", OUT_DIR)

    global_stable_ratios = read_global_stable_ratio_candidates(stability_root)

    if global_stable_ratios:
        print(f"Global stable candidate novel ratios loaded: {len(global_stable_ratios)}")
    else:
        print("No global stable candidate novel-ratio exploratory feature set will be used.")

    all_metrics = []
    all_classwise = []
    all_confusions = []
    all_feature_details = []

    for method in IMPUTATION_METHODS:
        method_data_dir = DATA_ROOT / method

        if not method_data_dir.exists():
            raise FileNotFoundError(f"Data directory was not found: {method_data_dir}")

        for fold in range(1, N_OUTER_FOLDS + 1):
            print(f"\n========== {method} fold {fold} ==========")

            train_path = (
                method_data_dir
                / f"fold_{fold:02d}_train_with_ratios.xlsx"
            )

            test_path = (
                method_data_dir
                / f"fold_{fold:02d}_test_with_ratios.xlsx"
            )

            if not train_path.exists():
                raise FileNotFoundError(f"Training fold was not found: {train_path}")

            if not test_path.exists():
                raise FileNotFoundError(f"Test fold was not found: {test_path}")

            champion_xlsx = find_champion_xlsx(
                champion_root,
                method,
                fold,
                RHO_TAG
            )

            champions, champ_df = read_champions(champion_xlsx)

            train_df = pd.read_excel(train_path)
            test_df = pd.read_excel(test_path)

            train_df.columns = [str(c).strip() for c in train_df.columns]
            test_df.columns = [str(c).strip() for c in test_df.columns]

            X_train, X_test, y_train, y_test, le, feature_cols = prepare_xy(
                train_df,
                test_df
            )

            feature_sets = build_feature_sets(
                feature_cols,
                champions,
                global_stable_ratios=global_stable_ratios
            )

            detail_rows = feature_set_detail_rows(method, fold, feature_sets)
            all_feature_details.extend(detail_rows)

            for fs_name, fs_features in feature_sets.items():
                fs_features = [f for f in fs_features if f in feature_cols]

                print(f"{fs_name}: {len(fs_features)} features")

                metrics, class_df, cm_df = evaluate_feature_set(
                    X_train,
                    y_train,
                    X_test,
                    y_test,
                    le,
                    fs_features,
                    fs_name
                )

                if metrics is None:
                    print(f"  Skipping empty feature set: {fs_name}")
                    continue

                metrics["Method"] = method
                metrics["Outer_fold"] = fold
                metrics["Train_file"] = str(train_path)
                metrics["Test_file"] = str(test_path)
                metrics["Champion_file"] = str(champion_xlsx)

                class_df["Method"] = method
                class_df["Outer_fold"] = fold

                cm_df["Method"] = method
                cm_df["Outer_fold"] = fold

                all_metrics.append(metrics)
                all_classwise.append(class_df)
                all_confusions.append(cm_df)

    metrics_df = pd.DataFrame(all_metrics)
    classwise_df = pd.concat(all_classwise, axis=0, ignore_index=True)
    confusion_df = pd.concat(all_confusions, axis=0, ignore_index=True)
    feature_detail_df = pd.DataFrame(all_feature_details)

    summary_df = summarize_metrics(metrics_df)
    classwise_summary_df = summarize_classwise(classwise_df)
    delta_df = paired_delta_tests(metrics_df)

    out_xlsx = OUT_DIR / "stable_novel_ratio_group_contribution_results.xlsx"

    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        metrics_df.to_excel(
            writer,
            sheet_name="outer_fold_raw_metrics",
            index=False
        )

        summary_df.to_excel(
            writer,
            sheet_name="summary_by_feature_set",
            index=False
        )

        classwise_df.to_excel(
            writer,
            sheet_name="classwise_raw_metrics",
            index=False
        )

        classwise_summary_df.to_excel(
            writer,
            sheet_name="classwise_summary",
            index=False
        )

        delta_df.to_excel(
            writer,
            sheet_name="paired_delta_tests",
            index=False
        )

        feature_detail_df.to_excel(
            writer,
            sheet_name="feature_set_details",
            index=False
        )

        confusion_df.to_excel(
            writer,
            sheet_name="confusion_matrices",
            index=False
        )

        if global_stable_ratios:
            pd.DataFrame({
                "Global_stable_ratio_candidate": global_stable_ratios,
                "Display_feature": [clean_display_name(f) for f in global_stable_ratios],
                "Ratio_group": [ratio_group(f) for f in global_stable_ratios]
            }).to_excel(
                writer,
                sheet_name="global_stable_ratio_list",
                index=False
            )

    print("\nResults saved to:", out_xlsx)

    macro_f1_fig = OUT_DIR / "feature_set_macro_f1_summary.png"
    plot_macro_f1_summary(summary_df, macro_f1_fig)

    delta_fig = OUT_DIR / "paired_delta_macro_f1_summary.png"
    plot_delta_summary(delta_df, delta_fig)

    print("Macro-F1 figure saved to:", macro_f1_fig.with_name(f"{macro_f1_fig.stem}_global_mean{macro_f1_fig.suffix}"))
    print("Macro-F1 figure saved to:", macro_f1_fig.with_name(f"{macro_f1_fig.stem}_knn{macro_f1_fig.suffix}"))
    print("Delta figure saved to:", delta_fig)

    print("\n========== summary_by_feature_set ==========")
    show_cols = [
        "Method",
        "Feature_set",
        "N_features_mean_plus_minus_std",
        "Accuracy_mean_plus_minus_std",
        "Balanced_accuracy_mean_plus_minus_std",
        "Macro_F1_mean_plus_minus_std"
    ]
    show_cols = [c for c in show_cols if c in summary_df.columns]
    print(summary_df[show_cols])

    print("\n========== paired_delta_tests: Macro-F1 ==========")
    if not delta_df.empty:
        print(delta_df[delta_df["Metric"] == "Macro_F1"][
            [
                "Method",
                "Comparison",
                "N_pairs",
                "Mean_delta",
                "SD_delta",
                "Paired_t_p",
                "Wilcoxon_p"
            ]
        ])

    print("\nAll tasks completed.")


# ============================================================
# 7. Program entry point
# ============================================================

if __name__ == "__main__":
    main()

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 0. Path settings
# ============================================================

def find_project_root():
    if "__file__" in globals():
        start_dir = Path(__file__).resolve().parent
    else:
        start_dir = Path.cwd().resolve()

    for path in [start_dir] + list(start_dir.parents):
        if (path / "Code").exists() and (path / "Data").exists() and (path / "Result").exists():
            return path

    raise FileNotFoundError(
        "Project root was not found. Please make sure the project contains Code, Data, and Result folders."
    )

PROJECT_ROOT = find_project_root()

INPUT_DIR = (
    PROJECT_ROOT
    / "Result"
    / "06_Stable_Novel_Ratio_Group_Contribution_Experiment"
)

INPUT_XLSX = INPUT_DIR / "stable_novel_ratio_group_contribution_results.xlsx"

# Compatibility fallback for earlier output naming.
if not INPUT_XLSX.exists():
    legacy_input = INPUT_DIR / "06_stable_novel_ratio_group_contribution_results.xlsx"
    if legacy_input.exists():
        INPUT_XLSX = legacy_input

OUT_DIR = INPUT_DIR / "Feature_Contribution_Figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1. Read input data
# ============================================================

summary = pd.read_excel(INPUT_XLSX, sheet_name="summary_by_feature_set")
delta = pd.read_excel(INPUT_XLSX, sheet_name="paired_delta_tests")

print("summary_by_feature_set:")
print(summary.head())

print("\npaired_delta_tests:")
print(delta.head())


# ============================================================
# 2. Name mapping
# ============================================================

method_map = {
    "global_mean": "global-mean",
    "knn": "KNN"
}

feature_label_map = {
    "Full_features": "Full features",
    "Champions_rho0.90": "Cluster champions",
    "Classical_features_only": "Classical only",
    "Fold_novel_champion_ratios_only": "Stable novel ratios only",
    "Classical_plus_fold_novel_champions": "Classical + novel ratios",
    "Champions_without_fold_novel_ratios": "Champions without novel ratios",
    "Global_stable_ratio_candidates_only_exploratory": "Global stable ratios only",
    "Classical_plus_global_stable_ratios_exploratory": "Classical + global stable ratios"
}

core_order = [
    "Classical only",
    "Stable novel ratios only",
    "Classical + novel ratios",
    "Champions without novel ratios",
    "Cluster champions",
    "Full features"
]

all_order = [
    "Classical only",
    "Stable novel ratios only",
    "Classical + novel ratios",
    "Champions without novel ratios",
    "Cluster champions",
    "Full features",
    "Global stable ratios only",
    "Classical + global stable ratios"
]

summary["Method_label"] = summary["Method"].map(method_map).fillna(summary["Method"])
summary["Feature_label"] = summary["Feature_set"].map(feature_label_map).fillna(summary["Feature_set"])


# ============================================================
# 3. Save performance table
# ============================================================

def resolve_column(df, candidates):
    """
    Return the first existing column from candidate column names.
    """
    for col in candidates:
        if col in df.columns:
            return col

    raise KeyError(
        f"None of the candidate columns were found: {candidates}. "
        f"Available columns: {df.columns.tolist()}"
    )


n_features_col = resolve_column(
    summary,
    ["N_features_mean_plus_minus_std", "N_features_mean±std"]
)

accuracy_col = resolve_column(
    summary,
    ["Accuracy_mean_plus_minus_std", "Accuracy_mean±std"]
)

balanced_accuracy_col = resolve_column(
    summary,
    ["Balanced_accuracy_mean_plus_minus_std", "Balanced_accuracy_mean±std"]
)

macro_f1_col = resolve_column(
    summary,
    ["Macro_F1_mean_plus_minus_std", "Macro_F1_mean±std"]
)

performance_table_cols = [
    "Method_label",
    "Feature_label",
    n_features_col,
    accuracy_col,
    balanced_accuracy_col,
    macro_f1_col
]

performance_table = summary[performance_table_cols].copy()
performance_table.columns = [
    "Imputation workflow",
    "Feature set",
    "No. features",
    "Accuracy",
    "Balanced accuracy",
    "Macro-F1"
]

performance_table["Feature set"] = pd.Categorical(
    performance_table["Feature set"],
    categories=all_order,
    ordered=True
)

performance_table["Imputation workflow"] = pd.Categorical(
    performance_table["Imputation workflow"],
    categories=["global-mean", "KNN"],
    ordered=True
)

performance_table = performance_table.sort_values(
    ["Imputation workflow", "Feature set"]
).reset_index(drop=True)

table_path = OUT_DIR / "table_feature_set_performance.xlsx"

performance_table.to_excel(table_path, index=False)
print("\nPerformance table saved to:", table_path)


# ============================================================
# 4. Global plotting parameters
# ============================================================

plt.rcParams["font.family"] = "Arial"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["axes.linewidth"] = 1.4
plt.rcParams["xtick.major.width"] = 1.3
plt.rcParams["ytick.major.width"] = 1.3


# ============================================================
# 5. Automatically calculate x-axis limits
# ============================================================

def auto_xlim_macro_f1(data, feature_order, margin=0.015):
    plot_df = data[data["Feature_label"].isin(feature_order)].copy()

    min_v = float((plot_df["Macro_F1_mean"] - plot_df["Macro_F1_std"]).min())
    max_v = float((plot_df["Macro_F1_mean"] + plot_df["Macro_F1_std"]).max())

    x_min = max(0.0, min_v - margin)
    x_max = min(1.0, max_v + margin)

    # Keep the x-axis ranges broadly consistent across figures.
    x_min = min(x_min, 0.75)
    x_max = max(x_max, 0.93)

    return x_min, x_max


def auto_xlim_delta(data, margin_ratio=0.18):
    min_v = float((data["Mean_delta"] - data["SD_delta"]).min())
    max_v = float((data["Mean_delta"] + data["SD_delta"]).max())
    margin = max(0.01, (max_v - min_v) * margin_ratio)

    return min_v - margin, max_v + margin


# ============================================================
# 6. Horizontal dot plot function
#    The legend is placed on the right side to avoid overlap.
# ============================================================

def plot_feature_set_dotplot(
    data,
    feature_order,
    out_path,
    title="Feature-set comparison",
    figsize=(10.8, 6.2),
    right_margin=0.78
):
    plot_df = data[data["Feature_label"].isin(feature_order)].copy()

    plot_df["Feature_label"] = pd.Categorical(
        plot_df["Feature_label"],
        categories=feature_order,
        ordered=True
    )

    plot_df = plot_df.sort_values("Feature_label")

    fig, ax = plt.subplots(figsize=figsize)

    y_positions = np.arange(len(feature_order))

    method_styles = {
        "global-mean": {
            "offset": -0.13,
            "marker": "o",
            "label": "global-mean"
        },
        "KNN": {
            "offset": 0.13,
            "marker": "s",
            "label": "KNN"
        }
    }

    for method, style in method_styles.items():
        sub = plot_df[plot_df["Method_label"] == method].set_index("Feature_label")

        xs, xerrs, ys = [], [], []

        for i, feat in enumerate(feature_order):
            if feat in sub.index:
                row = sub.loc[feat]
                xs.append(float(row["Macro_F1_mean"]))
                xerrs.append(float(row["Macro_F1_std"]))
                ys.append(i + style["offset"])

        ax.errorbar(
            xs,
            ys,
            xerr=xerrs,
            fmt=style["marker"],
            markersize=8.5,
            capsize=4,
            linewidth=2.0,
            elinewidth=1.8,
            markeredgewidth=1.2,
            label=style["label"]
        )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(feature_order, fontsize=13, fontweight="bold")

    ax.set_xlabel("Outer-test Macro-F1", fontsize=15, fontweight="bold")
    ax.set_ylabel("Feature set", fontsize=15, fontweight="bold")
    ax.set_title(title, fontsize=17, fontweight="bold", pad=12)

    ax.set_xlim(*auto_xlim_macro_f1(data, feature_order))

    ax.grid(axis="x", linestyle="--", alpha=0.45)

    ax.legend(
        frameon=False,
        fontsize=13,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        borderaxespad=0.0,
        handlelength=2.2
    )

    ax.tick_params(axis="x", labelsize=13, width=1.3)
    for tick in ax.get_xticklabels():
        tick.set_fontweight("bold")

    for spine in ax.spines.values():
        spine.set_linewidth(1.4)

    ax.invert_yaxis()

    plt.tight_layout(rect=[0, 0, right_margin, 1])

    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Figure saved to:", out_path)


# ============================================================
# 7. Core feature-set Macro-F1 figure
# ============================================================

fig9a_path = OUT_DIR / "fig9a_feature_set_macro_f1_core_clean.png"

plot_feature_set_dotplot(
    data=summary,
    feature_order=core_order,
    out_path=fig9a_path,
    title="Macro-F1 of Main Feature Sets",
    figsize=(10.8, 6.2),
    right_margin=0.78
)


# ============================================================
# 8. All feature-set Macro-F1 figure
# ============================================================

figS_path = OUT_DIR / "figS_feature_set_macro_f1_all_clean.png"

plot_feature_set_dotplot(
    data=summary,
    feature_order=all_order,
    out_path=figS_path,
    title="Macro-F1 of All Feature Sets",
    figsize=(11.3, 7.4),
    right_margin=0.78
)


# ============================================================
# 9. Paired-difference figure
# ============================================================

delta_macro = delta[delta["Metric"] == "Macro_F1"].copy()
delta_macro["Method_label"] = delta_macro["Method"].map(method_map).fillna(delta_macro["Method"])

comparison_label_map = {
    "Classical_plus_fold_novel_minus_Classical":
        "Classical + novel\nminus Classical",
    "Novel_only_minus_Classical":
        "Novel only\nminus Classical",
    "Champions_minus_Champions_without_novel":
        "Champions\nminus Champions without novel",
    "Classical_plus_global_stable_minus_Classical_exploratory":
        "Classical + global stable\nminus Classical",
    "Global_stable_only_minus_Classical_exploratory":
        "Global stable only\nminus Classical"
}

delta_macro["Comparison_label"] = (
    delta_macro["Comparison"]
    .map(comparison_label_map)
    .fillna(delta_macro["Comparison"])
)

core_delta_order = [
    "Classical + novel\nminus Classical",
    "Novel only\nminus Classical",
    "Champions\nminus Champions without novel"
]

delta_plot = delta_macro[
    delta_macro["Comparison_label"].isin(core_delta_order)
].copy()

delta_plot["Comparison_label"] = pd.Categorical(
    delta_plot["Comparison_label"],
    categories=core_delta_order,
    ordered=True
)

delta_plot = delta_plot.sort_values("Comparison_label")

fig, ax = plt.subplots(figsize=(10.8, 5.7))

y_positions = np.arange(len(core_delta_order))

method_styles = {
    "global-mean": {
        "offset": -0.13,
        "marker": "o",
        "label": "global-mean"
    },
    "KNN": {
        "offset": 0.13,
        "marker": "s",
        "label": "KNN"
    }
}

for method, style in method_styles.items():
    sub = delta_plot[delta_plot["Method_label"] == method].set_index("Comparison_label")

    xs, xerrs, ys = [], [], []

    for i, comp in enumerate(core_delta_order):
        if comp in sub.index:
            row = sub.loc[comp]
            xs.append(float(row["Mean_delta"]))
            xerrs.append(float(row["SD_delta"]))
            ys.append(i + style["offset"])

    ax.errorbar(
        xs,
        ys,
        xerr=xerrs,
        fmt=style["marker"],
        markersize=8.5,
        capsize=4,
        linewidth=2.0,
        elinewidth=1.8,
        markeredgewidth=1.2,
        label=style["label"]
    )

ax.axvline(0, linewidth=1.5, linestyle="--")

ax.set_yticks(y_positions)
ax.set_yticklabels(core_delta_order, fontsize=13, fontweight="bold")

ax.set_xlabel("Paired Delta Macro-F1", fontsize=15, fontweight="bold")
ax.set_ylabel("Comparison", fontsize=15, fontweight="bold")
ax.set_title("Paired Contribution of Stable Novel Ratios", fontsize=17, fontweight="bold", pad=12)

ax.set_xlim(*auto_xlim_delta(delta_plot))

ax.grid(axis="x", linestyle="--", alpha=0.45)

ax.legend(
    frameon=False,
    fontsize=13,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    borderaxespad=0.0,
    handlelength=2.2
)

ax.tick_params(axis="x", labelsize=13, width=1.3)
for tick in ax.get_xticklabels():
    tick.set_fontweight("bold")

for spine in ax.spines.values():
    spine.set_linewidth(1.4)

ax.invert_yaxis()

plt.tight_layout(rect=[0, 0, 0.78, 1])

fig9b_path = OUT_DIR / "fig9b_paired_delta_macro_f1_clean.png"

plt.savefig(fig9b_path, dpi=300, bbox_inches="tight")
plt.show()

print("Paired-difference figure saved to:", fig9b_path)


# ============================================================
# 10. Combined two-panel figure
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 6.4))

# ---------- Panel a ----------
ax = axes[0]

plot_df = summary[summary["Feature_label"].isin(core_order)].copy()
plot_df["Feature_label"] = pd.Categorical(
    plot_df["Feature_label"],
    categories=core_order,
    ordered=True
)
plot_df = plot_df.sort_values("Feature_label")

y_positions = np.arange(len(core_order))

method_styles = {
    "global-mean": {
        "offset": -0.13,
        "marker": "o",
        "label": "global-mean"
    },
    "KNN": {
        "offset": 0.13,
        "marker": "s",
        "label": "KNN"
    }
}

for method, style in method_styles.items():
    sub = plot_df[plot_df["Method_label"] == method].set_index("Feature_label")

    xs, xerrs, ys = [], [], []

    for i, feat in enumerate(core_order):
        if feat in sub.index:
            row = sub.loc[feat]
            xs.append(float(row["Macro_F1_mean"]))
            xerrs.append(float(row["Macro_F1_std"]))
            ys.append(i + style["offset"])

    ax.errorbar(
        xs,
        ys,
        xerr=xerrs,
        fmt=style["marker"],
        markersize=8.0,
        capsize=4,
        linewidth=1.9,
        elinewidth=1.7,
        markeredgewidth=1.1,
        label=style["label"]
    )

ax.set_yticks(y_positions)
ax.set_yticklabels(core_order, fontsize=12, fontweight="bold")

ax.set_xlabel("Outer-test Macro-F1", fontsize=14, fontweight="bold")
ax.set_ylabel("Feature set", fontsize=14, fontweight="bold")
ax.set_title("(a) Feature-set Performance", fontsize=16, fontweight="bold", pad=10)

ax.set_xlim(*auto_xlim_macro_f1(summary, core_order))
ax.grid(axis="x", linestyle="--", alpha=0.45)

ax.tick_params(axis="x", labelsize=12, width=1.3)
for tick in ax.get_xticklabels():
    tick.set_fontweight("bold")

for spine in ax.spines.values():
    spine.set_linewidth(1.3)

ax.invert_yaxis()


# ---------- Panel b ----------
ax = axes[1]

y_positions = np.arange(len(core_delta_order))

for method, style in method_styles.items():
    sub = delta_plot[delta_plot["Method_label"] == method].set_index("Comparison_label")

    xs, xerrs, ys = [], [], []

    for i, comp in enumerate(core_delta_order):
        if comp in sub.index:
            row = sub.loc[comp]
            xs.append(float(row["Mean_delta"]))
            xerrs.append(float(row["SD_delta"]))
            ys.append(i + style["offset"])

    ax.errorbar(
        xs,
        ys,
        xerr=xerrs,
        fmt=style["marker"],
        markersize=8.0,
        capsize=4,
        linewidth=1.9,
        elinewidth=1.7,
        markeredgewidth=1.1,
        label=style["label"]
    )

ax.axvline(0, linewidth=1.5, linestyle="--")

ax.set_yticks(y_positions)
ax.set_yticklabels(core_delta_order, fontsize=12, fontweight="bold")

ax.set_xlabel("Paired Delta Macro-F1", fontsize=14, fontweight="bold")
ax.set_ylabel("Comparison", fontsize=14, fontweight="bold")
ax.set_title("(b) Paired Contribution", fontsize=16, fontweight="bold", pad=10)

ax.set_xlim(*auto_xlim_delta(delta_plot))
ax.grid(axis="x", linestyle="--", alpha=0.45)

ax.tick_params(axis="x", labelsize=12, width=1.3)
for tick in ax.get_xticklabels():
    tick.set_fontweight("bold")

for spine in ax.spines.values():
    spine.set_linewidth(1.3)

ax.invert_yaxis()

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    frameon=False,
    fontsize=13,
    loc="center left",
    bbox_to_anchor=(0.93, 0.5),
    handlelength=2.2
)

plt.tight_layout(rect=[0, 0, 0.91, 1])

combined_path = OUT_DIR / "fig9_combined_feature_contribution_clean.png"

plt.savefig(combined_path, dpi=300, bbox_inches="tight")
plt.show()

print("Combined figure saved to:", combined_path)


# ============================================================
# 11. Save paired-difference table
# ============================================================

delta_table_path = OUT_DIR / "table_paired_delta_macro_f1.xlsx"

delta_macro.to_excel(delta_table_path, index=False)

print("Paired-difference table saved to:", delta_table_path)

print("\nAll tasks completed. Output directory:")
print(OUT_DIR)